# 03 — NCI‑60 Toy Drug‑Response: Predict a Drug’s Z‑Score from RNA‑seq
Downloads two small CellMiner datasets (official NIH) and runs a PCA+Ridge demo. File formats occasionally change; this notebook prints what it finds so you can adjust if needed.

In [ ]:
!pip -q install pandas numpy scikit-learn

In [ ]:
import os, zipfile, glob
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold, cross_val_score
import warnings; warnings.filterwarnings('ignore')
RANDOM_STATE = 0

In [ ]:
# 1) Download CellMiner datasets
!wget -q "https://discover.nci.nih.gov/cellminer/download/processeddataset/DTP_NCI60_ZSCORE.zip" -O zscore.zip
!wget -q "https://discover.nci.nih.gov/cellminer/download/processeddataset/nci60_RNA__RNA_seq_composite_expression.zip" -O rnaseq.zip
print('Downloaded zscore.zip and rnaseq.zip')

In [ ]:
# 2) Unzip helpers
def unzip(zip_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(out_dir)
    return [str(p) for p in Path(out_dir).rglob('*')]
zfiles = unzip('zscore.zip', 'zscore')
efiles = unzip('rnaseq.zip', 'rnaseq')
print('Z-score files found (sample):', zfiles[:5])
print('RNA-seq files found (sample):', efiles[:5])

In [ ]:
# 3) Locate tables
z_candidates = [p for p in zfiles if p.lower().endswith(('.txt','.csv','.tsv'))]
e_candidates = [p for p in efiles if p.lower().endswith(('.txt','.csv','.tsv'))]
z_candidates, e_candidates

In [ ]:
# 4) Smart read
def smart_read(path):
    if path.endswith('.csv'):  return pd.read_csv(path)
    if path.endswith('.tsv'):  return pd.read_csv(path, sep='\t')
    return pd.read_csv(path, sep='\t')
z = smart_read(z_candidates[0])
expr = smart_read(e_candidates[0])
display(z.head()); display(expr.head()); print(z.shape, expr.shape)

In [ ]:
# 5) Tidy: rows = cell lines
def tidy(df):
    df = df.copy()
    if df.columns[0].lower() in ('gene','genes','feature','drug','compound','name'):
        df = df.set_index(df.columns[0])
    return df.T
Z = tidy(z)
EX = tidy(expr)
print('Z (cells x drugs):', Z.shape)
print('EX (cells x genes):', EX.shape)
common_cells = Z.index.intersection(EX.index)
Zc = Z.loc[common_cells]
EXc = EX.loc[common_cells]
print('Common cells:', len(common_cells))

In [ ]:
# 6) Choose a drug
drug_name = None
for name in Zc.columns:
    if 'doxorubicin' in str(name).lower() or 'adriamycin' in str(name).lower():
        drug_name = name; break
if drug_name is None: drug_name = Zc.columns[0]
print('Modeling drug:', drug_name)
y = Zc[drug_name].astype(float).values
X = EXc.astype(float).values

In [ ]:
# 7) PCA + Ridge with CV
n_comp = int(min(30, X.shape[0]-5)) if X.shape[0] > 5 else max(1, X.shape[0]-1)
pipe = Pipeline([
    ('scaler', StandardScaler(with_mean=True)),
    ('pca', PCA(n_components=n_comp, random_state=RANDOM_STATE)),
    ('ridge', RidgeCV(alphas=np.logspace(-3, 3, 25), cv=5))
])
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
r2 = cross_val_score(pipe, X, y, cv=cv, scoring='r2')
print('5-fold CV R^2 mean±sd:', r2.mean(), '+/-', r2.std())

In [ ]:
# 8) Simple leave-one-out-style demo
pipe.fit(X, y)
test_idx = 0
pipe.fit(np.delete(X, test_idx, axis=0), np.delete(y, test_idx))
pred_single = pipe.predict(X[[test_idx]])[0]
print(f"True={y[test_idx]:.3f}  Pred={pred_single:.3f}  Cell={EXc.index[test_idx]}")

**Notes**
- Data source: CellMiner official download page (NIH). URLs may change.
- High-dimensional, tiny-N → regularization and dimensionality reduction help stabilize estimates.
- Treat this as a toy; real pipelines need QC, matching, feature selection, and careful validation.